# All Games Atari Training Notebook
This notebook sets up and trains PPO models on multiple Atari games using stable-baselines3 in Google Colab.

In [ ]:
# Install dependencies
!pip install torch gymnasium[atari] stable-baselines3==2.2.1 autorom[accept-rom-license] numpy opencv-python matplotlib

## Game Environment Setup (game_env.py)

In [ ]:
import gymnasium as gym
import ale_py
from gymnasium.wrappers import RecordVideo
import cv2
import numpy as np
from gymnasium import ObservationWrapper
from gymnasium.spaces import Box

# Custom wrapper to resize and grayscale observations
class ResizeAndGrayScale(ObservationWrapper):
    def __init__(self, env, shape=(84, 84)):
        super().__init__(env)
        self.shape = shape
        self.observation_space = Box(
            low=0, high=255, shape=(self.shape[0], self.shape[1], 1), dtype=np.uint8
        )

    def observation(self, obs):
        obs = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        obs = cv2.resize(obs, self.shape, interpolation=cv2.INTER_AREA)
        return np.expand_dims(obs, -1).astype(np.uint8)

# Function to create a game environment
def make_game_env(game_name, record=False):
    env = gym.make(f'ALE/{game_name}-v5', render_mode='rgb_array')
    env = ResizeAndGrayScale(env)
    if record:
        env = RecordVideo(env, video_folder='/content/videos', episode_trigger=lambda ep: True)
    return env

# Test the environment
test_env = make_game_env('Pong', record=False)
obs = test_env.reset()
print('Observation shape:', obs[0].shape)
test_env.close()

## Training Logic (training.py)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
import os
import matplotlib.pyplot as plt
import numpy as np

# Custom callback for training progress tracking
class TrainingTracker(BaseCallback):
    def __init__(self, game_name='Game', check_freq=1000, save_path='/content/training_plots'):
        super(TrainingTracker, self).__init__(verbose=1)
        self.game_name = game_name
        self.check_freq = check_freq
        self.save_path = save_path
        self.episode_rewards = []
        self.episode_lengths = []
        self.advantages = []
        self.win_statuses = []
        self.losses = {'policy_loss': [], 'value_loss': [], 'entropy_loss': []}
        self.total_steps = 0

    def _on_step(self) -> bool:
        self.total_steps += 1
        if self.total_steps % self.check_freq == 0:
            mean_reward = np.mean(self.episode_rewards[-10:]) if self.episode_rewards else 0
            win_rate = np.mean(self.win_statuses[-10:]) if self.win_statuses else 0
            print(f'Step: {self.total_steps}, Mean Reward: {mean_reward:.2f}, Win Rate: {win_rate:.2%}')
        return True

    def _on_rollout_end(self) -> None:
        # Collect episode info from rollout buffer
        buffer = self.locals.get('rollout_buffer')
        if buffer is not None:
            ep_rew = buffer.rewards.sum()
            ep_len = len(buffer.rewards)
            ep_adv = buffer.advantages.mean() if buffer.advantages is not None else 0
            win_status = ep_rew > 0

            self.episode_rewards.append(ep_rew)
            self.episode_lengths.append(ep_len)
            self.advantages.append(ep_adv)
            self.win_statuses.append(win_status)

        # Log losses if available
        if hasattr(self.model, 'logger'):
            log_dict = self.model.logger.get_log_dict()
            self.losses['policy_loss'].append(log_dict.get('train/policy_loss', 0))
            self.losses['value_loss'].append(log_dict.get('train/value_loss', 0))
            self.losses['entropy_loss'].append(log_dict.get('train/entropy_loss', 0))

    def plot_training_results(self):
        os.makedirs(self.save_path, exist_ok=True)
        plt.figure(figsize=(15, 5))

        # Plot rewards
        plt.subplot(1, 3, 1)
        plt.plot(self.episode_rewards, label='Episode Reward')
        plt.xlabel('Episode')
        plt.ylabel('Reward')
        plt.title(f'{self.game_name} Rewards')
        plt.legend()

        # Plot advantages
        plt.subplot(1, 3, 2)
        plt.plot(self.advantages, label='Mean Advantage')
        plt.xlabel('Episode')
        plt.ylabel('Advantage')
        plt.title(f'{self.game_name} Advantages')
        plt.legend()

        # Plot win status
        plt.subplot(1, 3, 3)
        plt.plot(self.win_statuses, label='Win Status')
        plt.xlabel('Episode')
        plt.ylabel('Win (1) / Loss (0)')
        plt.title(f'{self.game_name} Win Rate')
        plt.legend()

        plt.tight_layout()
        plt.savefig(f'{self.save_path}/{self.game_name}_training_plot.png')
        plt.close()

# Function to create the PPO model
def make_ppo_model(env, action_type='discrete'):
    return PPO(
        'CnnPolicy',
        env,
        verbose=1,
        device='cuda',
        n_steps=128,
        batch_size=256,
        n_epochs=4,
        learning_rate=2.5e-4,
    )

# Function to train the model with tracking
def train_ppo_with_tracking(env, model, total_timesteps, game_name='Game'):
    tracker = TrainingTracker(game_name=game_name)
    model.learn(total_timesteps=total_timesteps, callback=tracker, log_interval=10)
    tracker.plot_training_results()
    return model

## Training Multiple Games

In [ ]:
# List of games to train
games = ['Pong', 'Breakout', 'SpaceInvaders']
total_timesteps = 100000  # Adjust as needed (e.g., 1M for better results)

# Ensure directories exist
os.makedirs('/content/training_plots', exist_ok=True)
os.makedirs('/content/videos', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)

# Train each game
for game in games:
    print(f'\nTraining {game}...')
    env = make_game_env(game, record=False)  # Set record=True for video
    model = make_ppo_model(env)
    model = train_ppo_with_tracking(env, model, total_timesteps, game_name=game)
    model.save(f'/content/models/ppo_{game.lower()}')
    print(f'{game} training complete! Model saved as ppo_{game.lower()}.zip')
    env.close()

## Test a Trained Model

In [ ]:
# Test a model (e.g., Pong)
game_to_test = 'Pong'
env = make_game_env(game_to_test, record=True)  # Record test episode
model = PPO.load(f'/content/models/ppo_{game_to_test.lower()}')

obs = env.reset()[0]
done = False
total_reward = 0

while not done:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    total_reward += reward

print(f'Total Reward for {game_to_test}: {total_reward} 0:.2f}')
env.close()